<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/03_Advanced/L34_Mixed_Precision_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L34: Mixed Precision Training - FP16/BF16 and AMP

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Advanced  
**Lesson:** 34 of 45

---

## Learning Objectives

By the end of this lesson, you will:
- Enable automatic mixed precision (AMP) in training
- Understand FP16 vs BF16 and gradient scaling
- Use Trainer fp16/bf16 and compare memory and speed

---

## Concept: Mixed Precision

**Mixed precision**: compute in FP16 or BF16, keep master weights in FP32 (or FP16). Reduces memory and can speed up on Tensor Cores. **Gradient scaling** prevents underflow in FP16. PyTorch autocast + GradScaler; Trainer sets fp16=True or bf16=True.

---

In [ ]:
!pip install transformers torch accelerate datasets -q
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Trainer with FP16

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

train_ds = load_dataset("imdb", split="train", trust_remote_code=True).select(range(300))
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
args = TrainingArguments(
    output_dir="./out_amp_l34",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()
print("FP16 reduces memory and can speed up on compatible GPUs.")

## Part 2: BF16 (Better for Stability)

---

In [ ]:
# BF16 has same exponent range as FP32; often no gradient scaling needed.
args_bf16 = TrainingArguments(
    output_dir="./out_bf16_l34",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    bf16=torch.cuda.is_available(),
    report_to="none",
)
print("Use bf16=True on Ampere+ GPUs; more stable than fp16, no scaler needed.")

## Part 3: Manual Autocast (Concept)

---

In [ ]:
# In a custom loop:
# with torch.autocast(device_type='cuda', dtype=torch.float16):
#     loss = model(**batch).loss
# scaler.scale(loss).backward()
# scaler.step(optimizer)
# scaler.update()
print("Trainer handles autocast and GradScaler when fp16=True.")

## Exercises

1. Compare peak GPU memory: fp16=False vs fp16=True (same batch size).
2. Try bf16 on an A100/Ampere GPU and compare loss curve to fp32.
3. Increase batch size with fp16 until OOM; note the gain.

---

## Key Takeaways

1. FP16/BF16 reduce memory and leverage Tensor Cores; Trainer uses AMP automatically.
2. FP16 may need gradient scaling; BF16 is often more stable.
3. Use bf16 on Ampere+ when available.

---

## Next Lesson

**L35: Gradient Accumulation** – Simulating larger batches.

---